# Character networks: structure, importance, and removal
<a href="https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/barbie_network_analysis_2026.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

Run cells from the top; use the controls. No coding required.

The Barbie edge lists come from the 2025 teaching notebook. Counts have not been verified against the screenplay. Learn the method here; verify scenes before making claims about the film. Co-presence means shared scenes, not screen time. Dialogue means speaker → addressee.


In [ ]:
# Colab includes these libraries; no package installation is needed there.
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import io
from IPython.display import display

plt.rcParams.update({'figure.dpi': 110})
print('NetworkX', nx.__version__)

import ipywidgets as widgets


## Load the teaching network
Leave `USE_UPLOADS = False` for Barbie. For your own project later, switch it to True and upload two CSVs with `Source,Target,Weight` columns. Consistent names and documented counting rules matter; the notebook does not extract relationships from a PDF.


In [ ]:
# ==============================================================================
# CO-PRESENCE DATA (undirected: A shares scene with B)
# ==============================================================================
copresence_data = """Source,Target,Weight
Barbie Margot,Ken Ryan Gosling,20
Barbie Margot,Gloria,12
Barbie Margot,Sasha,11
Barbie Margot,Weird Barbie,9
Barbie Margot,Ken Simu Liu,8
Barbie Margot,Mattel CEO,6
Barbie Margot,President Barbie,5
Barbie Margot,Allan,4
Barbie Margot,Ruth,3
Ken Ryan Gosling,Ken Simu Liu,10
Ken Ryan Gosling,President Barbie,7
Ken Ryan Gosling,Doctor Barbie,6
Ken Ryan Gosling,Physicist Barbie,6
Ken Ryan Gosling,Ken Kingsley,5
Ken Ryan Gosling,Gloria,4
Ken Ryan Gosling,Sasha,4
Ken Ryan Gosling,Mattel CEO,4
Ken Ryan Gosling,Weird Barbie,2
Gloria,Sasha,8
Gloria,Mattel CEO,5
Gloria,Weird Barbie,3
Gloria,Nerdy Well Meaning Dad,3
Sasha,Nerdy Well Meaning Dad,3
Mattel CEO,Aaron (Mattel Employee),4
Mattel CEO,Other Board Members,3
President Barbie,Doctor Barbie,5
President Barbie,Physicist Barbie,5
President Barbie,Judge Barbie,5
Doctor Barbie,Physicist Barbie,4
Doctor Barbie,Judge Barbie,4
Physicist Barbie,Judge Barbie,4
Ken Simu Liu,Ken Kingsley,4
Ken Simu Liu,Ken Ncuti,4
Ken Simu Liu,Ken Scott,4
Allan,Skipper,3
Allan,Midge,2
Allan,Weird Barbie,4
Skipper,Midge,2
Midge,Allan,1
Judge Barbie,Journalist Barbie,2
"""

# ==============================================================================
# DIALOGUE DATA (directed: Source speaks TO Target)
# ==============================================================================
dialogue_data = """Source,Target,Weight
Barbie Margot,Ken Ryan Gosling,35
Barbie Margot,Gloria,25
Barbie Margot,Sasha,15
Barbie Margot,Weird Barbie,12
Barbie Margot,Mattel CEO,8
Barbie Margot,Ruth,6
Barbie Margot,President Barbie,5
Barbie Margot,Allan,4
Ken Ryan Gosling,Barbie Margot,40
Ken Ryan Gosling,Ken Simu Liu,12
Ken Ryan Gosling,Mattel CEO,3
Ken Ryan Gosling,Ken Ncuti,3
Ken Ryan Gosling,Ken Kingsley,3
Gloria,Barbie Margot,30
Gloria,Sasha,10
Gloria,Nerdy Well Meaning Dad,5
Gloria,Mattel CEO,4
Sasha,Barbie Margot,10
Sasha,Gloria,5
Sasha,Nerdy Well Meaning Dad,3
Weird Barbie,Barbie Margot,15
Weird Barbie,Allan,5
Mattel CEO,Barbie Margot,10
Mattel CEO,Gloria,5
Mattel CEO,Aaron (Mattel Employee),4
President Barbie,Ken Ryan Gosling,5
President Barbie,Doctor Barbie,3
President Barbie,Judge Barbie,3
Ruth,Barbie Margot,10
Allan,Weird Barbie,4
Allan,Gloria,3
Ken Simu Liu,Ken Ryan Gosling,10
Ken Simu Liu,Ken Ncuti,4
Doctor Barbie,President Barbie,3
Physicist Barbie,President Barbie,2
Judge Barbie,President Barbie,2
Skipper,Allan,1
Midge,Skipper,1
"""

# Set True only when you have two CSV edge lists ready.
USE_UPLOADS = False
if USE_UPLOADS:
    from google.colab import files
    print('Upload co-presence CSV first (one file).')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one co-presence CSV.')
    copresence_data = next(iter(uploaded.values())).decode('utf-8-sig')
    print('Upload dialogue CSV next (one file).')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one dialogue CSV.')
    dialogue_data = next(iter(uploaded.values())).decode('utf-8-sig')

def load_edges(text, directed=False):
    frame = pd.read_csv(io.StringIO(text))
    if not {'Source', 'Target', 'Weight'}.issubset(frame.columns):
        raise ValueError('CSV needs Source,Target,Weight columns.')
    frame = frame[['Source', 'Target', 'Weight']].copy()
    if frame.empty or frame.isna().any().any():
        raise ValueError('Edge list is empty or contains missing values.')
    for col in ['Source', 'Target']:
        frame[col] = frame[col].astype(str).str.strip()
        if frame[col].eq('').any():
            raise ValueError('Character names cannot be blank.')
    frame['Weight'] = pd.to_numeric(frame['Weight'], errors='raise')
    if not np.isfinite(frame['Weight']).all() or (frame['Weight'] <= 0).any():
        raise ValueError('Weights must be finite positive numbers.')
    if (frame['Source'] == frame['Target']).any():
        raise ValueError('Remove self-links before running.')
    if not directed:
        pairs = [sorted([a, b]) for a, b in zip(frame.Source, frame.Target)]
        frame[['Source', 'Target']] = pairs
    if frame.duplicated(['Source', 'Target']).any():
        print('Repeated pairs combined by summing weights. Check for accidental double counting.')
    frame = frame.groupby(['Source', 'Target'], as_index=False)['Weight'].sum()
    graph = nx.from_pandas_edgelist(frame, 'Source', 'Target', edge_attr='Weight',
                                   create_using=nx.DiGraph() if directed else nx.Graph())
    return frame, graph

df_copresence, G = load_edges(copresence_data)
df_dialogue, G_dialogue = load_edges(dialogue_data, directed=True)
print(f'Co-presence: {len(G)} characters, {G.number_of_edges()} relationships')
print(f'Dialogue: {len(G_dialogue)} characters, {G_dialogue.number_of_edges()} directed relationships')
display(df_copresence.head())


## 1. What structure do you see?
Look for hubs, clusters, bridges, and peripheral characters. Is there evidence for a hub-and-spoke, core-periphery, or several-community structure? Explain one observation before looking at the rankings. A layout alone does not establish a small-world or power-law network.


In [ ]:
# === Community Detection ===
groups = nx.community.louvain_communities(G, weight='Weight', seed=42)
partition = {node: cid for cid, members in enumerate(groups) for node in members}
num_communities = max(partition.values()) + 1
cmap = plt.get_cmap('Set2', num_communities)
node_colors = [cmap(partition[n]) for n in G.nodes()]

# === Visualization ===
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.8, seed=42)
edge_weights = [G[u][v]['Weight'] * 0.4 for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=700, node_color=node_colors, alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, edge_color='gray')
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

plt.title("Visualization 1: Co-Presence Network with Communities", fontsize=14, fontweight='bold')
plt.figtext(0.5, 0.01, 'Color = detected community; line width = shared-scene count. Positions are a layout, not geography.', ha='center', fontsize=9)
plt.axis('off')
plt.tight_layout()
plt.show()

# Print communities
print("\nDetected Communities:")
communities = {}
for node, comm in partition.items():
    communities.setdefault(comm, []).append(node)
for cid, members in sorted(communities.items()):
    print(f"  Community {cid}: {', '.join(members)}")

## 2. Who is important—and in what sense?
- **Degree:** number of different partners. **Strength:** total shared-scene counts across partners.
- **Betweenness:** share of shortest paths passing through a character.
- **Closeness:** short paths to others, adjusted for reachable characters.
- **Eigenvector:** connections to other central characters.

Paths below are unweighted; eigenvector uses shared-scene weights. Larger/darker nodes rank higher within a panel; compare ranks, not the numerical scales of different measures.


In [ ]:
degree = dict(G.degree())
strength = dict(G.degree(weight='Weight'))
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
eigenvector = nx.eigenvector_centrality(G, weight='Weight', max_iter=5000)
metrics = {'Degree': degree, 'Betweenness': betweenness,
           'Closeness': closeness, 'Eigenvector': eigenvector}
centrality_table = pd.DataFrame(metrics)
centrality_table['Strength'] = pd.Series(strength)
centrality_table.index.name = 'Character'
display(centrality_table.sort_values('Degree', ascending=False).round(3))
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (name, values) in zip(axes.flat, metrics.items()):
    maximum = max(values.values()) or 1
    sizes = [250 + 1400 * values[node] / maximum for node in G]
    nx.draw_networkx(G, pos, ax=ax, node_size=sizes,
        node_color=[values[node] / maximum for node in G], cmap=plt.cm.YlOrRd,
        vmin=0, vmax=1, edge_color='#b8b8b8', font_size=7, edgecolors='#555')
    leaders = sorted(values, key=values.get, reverse=True)[:3]
    ax.set_title(name + '\nTop three: ' + ', '.join(leaders), fontsize=10)
    ax.axis('off')
fig.suptitle('Same network and positions; importance depends on the measure', fontsize=14)
fig.tight_layout()
plt.show()


Choose two characters whose rankings differ across measures. What makes each important? Use the table and a connection in the graph. Does degree tell the same story as strength?

## 3. Who matters when removed?
Predict before running: choose a well-connected character and a bridge. Remove each separately from the original graph. Who disconnects people, and who mainly removes many direct ties?

This removes a node and its links; it does not simulate rewriting the story or other characters adapting.


In [ ]:
original_nodes = len(G)

def removal_summary(graph):
    components = list(nx.connected_components(graph))
    # Fixed original denominator: removed and disconnected pairs contribute zero.
    # This avoids an apparent gain from simply dropping a hard-to-reach character.
    inverse_distances = sum(1 / distance
        for source, distances in nx.all_pairs_shortest_path_length(graph)
        for target, distance in distances.items() if target != source)
    return {'Nodes left': len(graph), 'Links left': graph.number_of_edges(),
            'Components': len(components),
            'Largest component': max(map(len, components), default=0),
            'Isolates': nx.number_of_isolates(graph),
            'Reach efficiency (original pairs)': inverse_distances / (original_nodes * (original_nodes - 1))
                if original_nodes > 1 else 0}

baseline = removal_summary(G)
rows = []
for character in G:
    remaining = G.copy()
    remaining.remove_node(character)
    result = removal_summary(remaining)
    result['Character removed'] = character
    result['Efficiency loss'] = baseline['Reach efficiency (original pairs)'] - result['Reach efficiency (original pairs)']
    rows.append(result)
removal_table = pd.DataFrame(rows).set_index('Character removed')
print('Original network:')
display(pd.DataFrame([baseline]).round(3))
print('Each row starts over from the original network:')
display(removal_table.sort_values(['Largest component', 'Efficiency loss'], ascending=[True, False]).round(3))

remove_choice = widgets.Dropdown(options=sorted(G), description='Remove:', style={'description_width': 'initial'})
remove_button = widgets.Button(description='Remove and compare', button_style='primary')
remove_output = widgets.Output()
def compare_removal(_):
    character = remove_choice.value
    remaining = G.copy()
    remaining.remove_node(character)
    with remove_output:
        from IPython.display import clear_output
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for ax, graph, title in [(axes[0], G, 'Original'), (axes[1], remaining, 'Removed: ' + character)]:
            nx.draw_networkx(graph, pos, ax=ax, node_size=450,
                node_color=[cmap(partition[node]) for node in graph],
                edge_color='#aaa', font_size=7, edgecolors='#555')
            ax.set_title(title)
            ax.axis('off')
            ax.set_xlim(axes[0].get_xlim())
            ax.set_ylim(axes[0].get_ylim())
        fig.tight_layout()
        plt.show()
        display(pd.DataFrame([baseline, removal_summary(remaining)], index=['Original', 'After removal']).round(3))
remove_button.on_click(compare_removal)
display(widgets.VBox([remove_choice, remove_button, remove_output]))


Explain one removal result. Did it match your prediction? Distinguish losing the removed character's own connections from breaking connections among the people who remain. Removing a character can split the network without removing the most edges.

## 4. Does the answer depend on what counts as a connection?
Use the slider to exclude weak co-presence ties. Compare with the full network. Are the same hubs and bridges still visible? A low-frequency tie can carry a key scene.


In [ ]:
threshold = widgets.IntSlider(value=0, min=0, max=int(df_copresence.Weight.max()), description='Keep weight >', continuous_update=False, style={'description_width': 'initial'})
filter_button = widgets.Button(description='Filter and compare')
filter_output = widgets.Output()
def compare_filter(_):
    filtered = G.copy()
    filtered.remove_edges_from([(a, b) for a, b, data in G.edges(data=True) if data['Weight'] <= threshold.value])
    with filter_output:
        from IPython.display import clear_output
        clear_output(wait=True)
        plt.figure(figsize=(12, 8))
        nx.draw_networkx(filtered, pos, node_size=450,
            node_color=[cmap(partition[node]) for node in filtered], edge_color='#aaa', font_size=7)
        plt.title(f'Keep shared-scene counts > {threshold.value}; retain isolated characters')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        display(pd.DataFrame([baseline, removal_summary(filtered)], index=['Full network', 'Filtered']).round(3))
filter_button.on_click(compare_filter)
display(widgets.VBox([threshold, filter_button, filter_output]))


## 5. Does direction change the story?
Compare number of partners (degree) with addressed turns (strength). Incoming = addressed by others; outgoing = addresses others. These measure recorded interactions, not motives or emotional importance.


In [ ]:
in_degree = dict(G_dialogue.in_degree())
out_degree = dict(G_dialogue.out_degree())
in_strength = dict(G_dialogue.in_degree(weight='Weight'))
out_strength = dict(G_dialogue.out_degree(weight='Weight'))
try:
    eig_d = nx.eigenvector_centrality(G_dialogue, weight='Weight', max_iter=5000)
except nx.PowerIterationFailedConvergence:
    eig_d = {node: np.nan for node in G_dialogue}
    print('Eigenvector did not converge; shown as missing, not zero.')

dialogue_df = pd.DataFrame({
    'Character': list(G_dialogue),
    'In-Degree': [in_degree[n] for n in G_dialogue],
    'Out-Degree': [out_degree[n] for n in G_dialogue],
    'In-Strength': [in_strength[n] for n in G_dialogue],
    'Out-Strength': [out_strength[n] for n in G_dialogue],
    'Dialogue Eigenvector': [eig_d[n] for n in G_dialogue]
})
dialogue_df['In/Out Strength Ratio'] = dialogue_df['In-Strength'].div(
    dialogue_df['Out-Strength'].replace(0, np.nan))
dialogue_df = dialogue_df.sort_values('In-Strength', ascending=False)
display(dialogue_df.round(3))

plt.figure(figsize=(14, 10))
pos_d = nx.spring_layout(G_dialogue, k=0.7, seed=42)
d_weights = [G_dialogue[u][v]['Weight'] * 0.12 for u, v in G_dialogue.edges()]

nx.draw_networkx_nodes(G_dialogue, pos_d, node_size=600, node_color='skyblue', alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G_dialogue, pos_d, width=d_weights, alpha=0.6, edge_color='darkblue',
                       arrowstyle='->', arrowsize=15, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G_dialogue, pos_d, font_size=8, font_weight='bold')

plt.title("Visualization 4: Directed Dialogue Network (Arrow = Speaker → Addressee)", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## Take away one finding
Make one claim about structure or a character's role. Support it with a graph, a numerical comparison, and a scene you would check. What does the network reveal that you had not noticed?

For your own text: document the edge rule, check character names, and verify sample counts against scenes. A co-presence network and a dialogue network answer different questions. A high-centrality character is not automatically the most emotionally or thematically important.

Source: adapted from [the 2025 course notebook](https://github.com/KatherineElkins/networkx_miniproject_3_programming-humanity_2025/blob/main/IPHS200_Network_Analysis_Complete.ipynb). The inherited Barbie counts remain unverified.
